# Lab 2: Fine-Tuning a Small BERT Model to Route Complaints

**Time:** about 60 minutes

Every message that reaches the Fund has to end up with the right team: claims, compliance, ICT, member services.
Today someone reads each one and forwards it by hand. You'll build a model that does the first sort.

### From bags of words to transformers

In Lab 1 you used TF-IDF (term frequency-inverse document frequency), which treats a message as a **bag of words**:
it counts which words appear, but forgets their order. "The employer paid, not the member" and "The member paid, not
the employer" look identical to it.

This lab uses a **transformer**: a kind of neural network that reads the whole message, word order included, and gives
each word a meaning that depends on the words around it. "Paid" in *"I was paid"* and *"I was not paid"* no longer looks
the same. Transformers are the technology behind BERT, ChatGPT, and the language models you'll use from Day 7.

**BERT** (Bidirectional Encoder Representations from Transformers) is a well-known transformer from Google that reads
text in both directions at once. **DistilBERT** is a smaller, faster version ("distilled" from BERT) that keeps most of
its accuracy.

### Pre-training and fine-tuning

You won't train a transformer from scratch; that takes weeks and enormous amounts of text. You'll take one that has
already read a large amount of text in about a hundred languages (including Swahili), and **fine-tune** it: a short
extra round of training on the Fund's labelled messages.

Think of it like hiring an experienced graduate who already reads English and Swahili well (pre-training), then giving
them a week of on-the-job training on the Fund's own categories (fine-tuning). Much faster than teaching someone to
read from scratch.

```
pre-trained multilingual DistilBERT ─► + a small classification layer ─► fine-tune on ~4,500 labelled messages
                                                                      ─► route new messages, or hand them to a person
```

### Words you'll meet in this lab

| Word | Plain meaning |
|---|---|
| **transformer** | a neural network that reads a whole sequence and works out how each word relates to the others |
| **pre-trained model** | a model already trained on a huge amount of general text, which you can download |
| **fine-tuning** | a short extra round of training on your own data, for your own task |
| **Hugging Face** | a company and website hosting thousands of pre-trained models, plus the `transformers` library |
| **token / sub-word** | a piece of a word from the model's fixed vocabulary; rare words are split into several |
| **token id** | the number that stands for a token |
| **padding** | adding blank tokens so all messages in a batch are the same length |
| **attention mask** | a list of 1s and 0s telling the model which tokens are real and which are padding |
| **logits** | the model's raw scores for each category, before they're turned into probabilities |
| **softmax** | turns raw scores into probabilities that add up to 1 |
| **confusion matrix** | a table of actual vs predicted categories, showing which ones get mixed up |
| **confidence threshold** | the minimum probability needed to route a message automatically |

## Where this lab fits

**What you've done so far.** In Lab 1 you turned members' messages into numbers with TF-IDF, classified their
sentiment with logistic regression, learned word embeddings, and found complaint themes over time. You saw the limits
of word counts: they ignore word order, and every word has one meaning whatever the sentence. On Day 3 you trained a
small neural network in PyTorch with a loop you wrote yourself.

**What you'll do now.** You'll fine-tune a pre-trained multilingual transformer (DistilBERT) to predict which of nine
categories each message belongs to, compare it honestly with a TF-IDF baseline, see where it gets confused, and build a
routing function that only sends a message to a team automatically when the model is confident enough.

**Why it matters.** Today every message is read and forwarded by hand. Routing even a share of them automatically,
accurately, gets members' problems to the right team faster and frees staff for the cases that need judgement.

**What happens next.** Lab 3 extracts facts (names, member numbers, claim references, dates, amounts) from officers'
notes. Day 7 moves from models that do one job (like this router) to large language models that do many jobs from a
written instruction. They're built from the same transformer pieces you meet here.

## Imports

As well as the familiar scikit-learn tools (for the TF-IDF baseline and the measures), this lab uses Hugging Face's
`transformers` library:

- `AutoTokenizer`: downloads the tokenizer that matches a model, the tool that splits text into the model's tokens;
- `AutoModelForSequenceClassification`: downloads a pre-trained model and adds a classification layer on top;
- `Trainer` and `TrainingArguments`: a ready-made version of the training loop you wrote by hand on Day 3;
- `DataCollatorWithPadding`: pads each batch of messages to the same length.

In [ ]:
import os
import time
import warnings

# Skip a slow CPU-core check on Windows 11, and hide library notices.
os.environ["LOKY_MAX_CPU_COUNT"] = str(os.cpu_count() - 1)
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, confusion_matrix

# Hugging Face transformers: pre-trained models, their tokenizers, and a ready-made training loop (Trainer).
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import Trainer, TrainingArguments, DataCollatorWithPadding

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 170)

Choose the model and check for a GPU (graphics processing unit). `MODEL_NAME` is the model's name on the Hugging Face
website; the first time you use it, about 540 MB is downloaded and kept on your machine for next time.

Fine-tuning a transformer is heavy work. On a GPU it takes a few minutes; on a CPU it can take much longer. So without
a GPU, `QUICK_RUN` switches on, and the lab trains on a smaller sample for one epoch. The results will be a little
weaker, but the lesson is the same.

In [ ]:
SEED = 42
torch.manual_seed(SEED)

# The pre-trained model to start from: a small BERT that has read text in about 100 languages.
MODEL_NAME = "distilbert-base-multilingual-cased"

# On a CPU, train on a sample so the lab finishes in reasonable time.
GPU = torch.cuda.is_available()
QUICK_RUN = not GPU

if GPU:
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU: none, so QUICK_RUN is on")

### Theory: attention, the idea inside every transformer

The key part of a transformer is called **attention**. For each word, the model asks: *"which other words in this
message help me understand this one?"*, and builds the word's meaning from a weighted mix of them.

Take *"My employer deducted contributions but the Fund says they were not paid"*. To understand **paid**, the most
useful words are *not*, *contributions* and *Fund*. Attention gives those high weights and words like *My* low weights:

```
paid  ← looks at:   not (0.35)   contributions (0.25)   Fund (0.15)   deducted (0.10)   ... others (small)
```

Every word does this at the same time, and the model repeats it over several layers, each building on the one before.
The result is a vector for each word that **depends on its context**: *paid* here gets a different vector from *paid* in
"I was paid on time". That's exactly what Lab 1's Word2Vec couldn't do.

The weights aren't written by anyone. They're learned during pre-training, by reading enormous amounts of text and
predicting hidden words from their surroundings.

> **Where we are.** The libraries and the model name are set, and `QUICK_RUN` is on if there's no GPU. First, the
> labelled messages.

## 1. Data and labels

The same feedback file as Lab 1, with duplicates removed. This time the target is the `category`: which kind of
problem the message is about, and so which team should get it.

In [ ]:
fb = pd.read_csv("member_feedback.csv")
fb = fb.drop_duplicates(subset=["text"]).reset_index(drop=True)
fb["category"].value_counts()

Neural networks work with numbers, not names, so give each category a number. Two dictionaries let you translate in both
directions: `label2id` (name → number) and `id2label` (number → name). `enumerate` hands out the numbers 0, 1, 2 ...
along with each name.

In [ ]:
# Models work with numbers, so give each category a number: 0, 1, 2 ...
labels = sorted(fb["category"].unique())

label2id = {}           # category name -> number
id2label = {}           # number -> category name
for number, name in enumerate(labels):
    label2id[name] = number
    id2label[number] = name

fb["label"] = fb["category"].map(label2id)      # a new column with each message's category number
id2label

Split three ways:

- **train** (70%) to learn from;
- **validation** (15%) to watch during training, so you can see whether the model is still improving;
- **test** (15%) to report at the end, and look at nothing else until then.

`train_test_split` only splits in two, so use it twice. With `QUICK_RUN` on, the training set is cut to 1,500 messages.

In [ ]:
# First: 70% train, 30% "rest".
train, rest = train_test_split(fb, test_size=0.3, random_state=SEED, stratify=fb["label"])

# Then split the rest in half: 15% validation, 15% test.
val, test = train_test_split(rest, test_size=0.5, random_state=SEED, stratify=rest["label"])

if QUICK_RUN:
    train = train.sample(1500, random_state=SEED)       # a smaller sample for CPU-only machines

print(f"train {len(train):,} | validation {len(val):,} | test {len(test):,}")

> **Where we are.** Each category has a number, and the messages are split into train (70%), validation (15%) and test
> (15%). Before fine-tuning anything, set the bar.

## 2. The baseline to beat

Same rule as every other day: before anything clever, see what a simple model does. TF-IDF and logistic regression,
exactly as in Lab 1, but predicting the category instead of the sentiment. `sublinear_tf=True` dampens the effect of a
word repeated many times in one message.

`time.perf_counter()` measures how long training takes, so you can compare the cost of the two approaches too.
**Macro F1** averages the F1 score over all categories, so the small ones count as much as the big ones.

In [ ]:
start = time.perf_counter()        # start the stopwatch

tfidf = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)),
    ("clf", LogisticRegression(max_iter=3000, class_weight="balanced")),
])
tfidf.fit(train["text"], train["label"])
tfidf_pred = tfidf.predict(test["text"])

tfidf_seconds = time.perf_counter() - start
tfidf_f1 = f1_score(test["label"], tfidf_pred, average="macro")
print(f"TF-IDF macro F1 on test: {tfidf_f1:.3f}  (trained in {tfidf_seconds:.1f}s)")

> **Where we are.** TF-IDF with logistic regression has given a macro F1 to beat, after about a second of training. Now
> for the transformer, starting with how it reads text.

## 3. How a transformer sees text: tokens

Transformers don't read words. They read **tokens**: pieces of words from a fixed vocabulary learned during
pre-training (about 120,000 pieces for this model). Common words are one token, rarer words are split into pieces. A
piece starting with `##` continues the word before it: `maternity` might become `mater` + `##nity`.

This is how a transformer copes with words it has never seen, including spelling mistakes: it can still read the
pieces.

Every model comes with its own tokenizer, and you must use the matching one. `AutoTokenizer.from_pretrained(MODEL_NAME)`
downloads it. Compare an English sentence with a Swahili sentence of similar meaning:

In [ ]:
# Download (first time only) the tokenizer that belongs to the model.
tok = AutoTokenizer.from_pretrained(MODEL_NAME)

english = "I submitted my maternity claim but I have not been paid"
swahili = "Nimewasilisha madai ya fao la uzazi lakini sijalipwa"

english_tokens = tok.tokenize(english)
swahili_tokens = tok.tokenize(swahili)

print(len(english.split()), "words ->", len(english_tokens), "tokens:", english_tokens)
print(len(swahili.split()), "words ->", len(swahili_tokens), "tokens:", swahili_tokens)

The Swahili sentence has fewer words but more tokens. The model's vocabulary was built mostly from languages with far
more text on the internet than Swahili, so Swahili words get chopped into more pieces. Two practical consequences:

- messages in Swahili use more of the model's maximum length, so set `max_length` with Swahili in mind;
- on Day 7, when you pay for LLMs (large language models) by the token, the same message costs more in Swahili than in
  English.

The model doesn't see the pieces as text, but as numbers (**token ids**): each token's position in the vocabulary.
Calling the tokenizer directly (`tok(english)`) gives those ids. The first and last are special markers: `[CLS]`
(start of message) and `[SEP]` (end).

In [ ]:
encoded = tok(english)
print(encoded["input_ids"])

Transformers have a maximum message length, and longer limits cost more memory and time. To choose a sensible limit,
count how many tokens each message has, and look at the 50th, 95th and 99th percentiles (the length that 50%, 95% and
99% of messages fit within).

In [ ]:
token_counts = []
for message in fb["text"]:
    token_counts.append(len(tok(message)["input_ids"]))     # number of tokens in this message

pd.Series(token_counts).describe(percentiles=[0.5, 0.95, 0.99]).round(0)

In [ ]:
# Long enough for nearly every message. Anything longer is cut off.
MAX_LEN = 128

> **Where we are.** You've seen how the tokenizer splits English and Swahili into sub-word tokens and ids, and chosen a
> maximum length of 128 tokens from the length distribution. Next, package the messages the way the training code
> expects.

## 4. Prepare the data for PyTorch

The same `Dataset` idea as Day 3: something that can say how many examples there are, and hand over example number
*i*. This time each example is a message's token ids plus its label. The class below turns every message into token
ids once, up front.

A Dataset class needs three parts:

- `__init__`: runs once when you create it; here it tokenises every message (`truncation=True` cuts anything longer
  than `MAX_LEN`);
- `__len__`: says how many examples there are;
- `__getitem__(i)`: hands over example `i` as a dictionary with its `input_ids`, its `attention_mask` (1 for each real
  token; padding will get 0) and its label.

Padding happens per batch (the `DataCollatorWithPadding` below), so short messages don't waste computation on padding
up to 128 tokens.

In [ ]:
class FeedbackDataset(torch.utils.data.Dataset):
    """Messages as token ids, with their labels, in the form the Hugging Face Trainer expects."""

    def __init__(self, table):
        # Tokenise every message once. truncation=True cuts anything longer than MAX_LEN.
        self.encodings = tok(list(table["text"]), truncation=True, max_length=MAX_LEN)
        self.labels = list(table["label"])

    def __len__(self):
        # How many examples?
        return len(self.labels)

    def __getitem__(self, i):
        # Example number i: its token ids (and attention mask), plus its label.
        item = {
            "input_ids": self.encodings["input_ids"][i],
            "attention_mask": self.encodings["attention_mask"][i],
            "labels": self.labels[i],
        }
        return item


# One dataset for each part of the split.
train_ds = FeedbackDataset(train)
val_ds = FeedbackDataset(val)
test_ds = FeedbackDataset(test)

print(len(train_ds), "training examples")

# Look at example 0.
example = train_ds[0]
print("keys:", list(example.keys()))
print("first 10 token ids:", example["input_ids"][:10])
print("label:", example["labels"], "=", id2label[example["labels"]])

> **Where we are.** The three splits are wrapped in `FeedbackDataset` objects, ready to be served in padded batches.
> Now load the pre-trained model and train it on the Fund's categories.

### Theory: what changes during fine-tuning, and why the steps are tiny

The pre-trained model has about 135 million weights that already encode a lot about language. Fine-tuning adds one new
layer on top (one output per category) and then trains **everything** a little more on your labelled messages.

The danger is damaging what the model already knows. Big steps on a few thousand messages could overwrite general
language knowledge with quirks of the training set. This is sometimes called **catastrophic forgetting**. That's why
fine-tuning uses:

- a **very small learning rate** (0.00005, about 20 times smaller than on Day 3);
- a **warm-up**: even smaller steps at first, while the new random layer settles down, so its early random errors
  don't push the pre-trained layers off course;
- **few epochs** (1-3): the model learns the task quickly because most of the knowledge is already there.

The same idea comes back on Day 8, where you fine-tune a large language model with an even more careful method (LoRA)
that leaves the original weights untouched.

## 5. Fine-tune

`AutoModelForSequenceClassification` loads the pre-trained model and adds a fresh classification layer with one output
per category. That layer starts random. It's the part that learns most, while the pre-trained layers adjust gently.

Passing `id2label` and `label2id` stores the category names inside the model, so later it can report "digital_services"
rather than just "4".

When you run the next cell you'll see a message listing weights that are **missing** or **unexpected**. That's
expected, not an error: the classification layer is new (so its weights are "missing" from the download), and the
parts of the original model used for pre-training aren't needed any more (so they're "unexpected").

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(labels),       # one output per category
    id2label=id2label,            # so the model can report category names, not just numbers
    label2id=label2id,
)

The Hugging Face `Trainer` runs the training loop you wrote by hand on Day 3: batches, forward pass, loss,
backpropagation, optimiser step, evaluation. On a GPU (graphics processing unit) it also uses 16-bit arithmetic
(`fp16`), which roughly halves memory and speeds things up.

The Trainer calls the function below after each epoch to report how the validation set is doing. It receives the
model's raw scores (**logits**: one per category per message) and the true labels. `argmax(axis=-1)` picks the
category with the highest score for each message.

In [ ]:
def compute_metrics(eval_pred):
    """Accuracy and macro F1 on the validation set."""
    logits, true_labels = eval_pred
    predicted = logits.argmax(axis=-1)           # the category with the highest score
    return {
        "accuracy": (predicted == true_labels).mean(),
        "macro_f1": f1_score(true_labels, predicted, average="macro"),
    }

Now the training settings. The important ones:

| Setting | Meaning |
|---|---|
| `num_train_epochs` | how many times to go through the training messages (3, or 1 on a CPU) |
| `per_device_train_batch_size=32` | messages per training step |
| `learning_rate=5e-5` | 0.00005: very small steps, so the pre-trained knowledge isn't destroyed |
| `weight_decay=0.01` | a gentle pull on the weights towards zero, to reduce overfitting |
| `warmup_steps` | start with even smaller steps for the first 10% of training, while the new layer settles |
| `eval_strategy="epoch"` | check the validation set after each epoch |
| `save_strategy="no"` | don't save copies of the model along the way |
| `report_to="none"` | don't send training logs to an online service |

The warm-up length is worked out from the number of batches: `np.ceil` rounds up, because a last, smaller batch still
counts.

In [ ]:
if QUICK_RUN:
    EPOCHS = 1
else:
    EPOCHS = 3

# Learning-rate warm-up: start with tiny steps for the first 10% of training.
batches_per_epoch = int(np.ceil(len(train_ds) / 32))
warmup = int(0.1 * batches_per_epoch * EPOCHS)

args = TrainingArguments(
    output_dir="bert_runs",           # where the Trainer may write files
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=32,   # messages per training batch
    per_device_eval_batch_size=64,
    learning_rate=5e-5,
    weight_decay=0.01,
    warmup_steps=warmup,
    eval_strategy="epoch",            # check the validation set after each epoch
    save_strategy="no",               # don't save checkpoints along the way
    logging_steps=50,
    fp16=GPU,                         # 16-bit arithmetic on a GPU
    report_to="none",                 # don't send logs to an online service
    seed=SEED,
)

# Put it all together: the model, the settings, the data, the padding, and the measures.
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=DataCollatorWithPadding(tok),   # pads each batch to its longest message
    compute_metrics=compute_metrics,
)

Train. While it runs you'll see a progress bar, the training loss every 50 steps, and the validation accuracy and
macro F1 after each epoch. The loss should fall and the validation scores should rise.

In [ ]:
start = time.perf_counter()
trainer.train()
bert_seconds = time.perf_counter() - start
print(f"fine-tuning took {bert_seconds / 60:.1f} minutes")

> **Where we are.** The model is fine-tuned, and you watched the validation scores improve after each epoch. The test set
> hasn't been touched yet.

## 6. The honest comparison

Now, for the first time, use the test set. `trainer.predict` gives the logits for every test message. **Softmax**
turns each message's logits into probabilities that add up to 1: for example, 0.91 for "delayed_benefit", 0.05 for
"pension_payment" and small amounts for the rest. The predicted category is the one with the highest probability.

In [ ]:
# Raw scores (logits) for every test message: one row per message, one column per category.
output = trainer.predict(test_ds)
logits = torch.tensor(output.predictions, dtype=torch.float32)

# softmax turns raw scores into probabilities that add up to 1.
probs = torch.softmax(logits, dim=-1).numpy()

# The predicted category is the one with the highest probability.
bert_pred = probs.argmax(axis=1)

Put the two models side by side: quality (macro F1 and accuracy) and cost (training time).

In [ ]:
bert_f1 = f1_score(test["label"], bert_pred, average="macro")

comparison = pd.DataFrame({
    "macro F1": [tfidf_f1, bert_f1],
    "accuracy": [(tfidf_pred == test["label"]).mean(), (bert_pred == test["label"]).mean()],
    "training time (s)": [tfidf_seconds, bert_seconds],
}, index=["TF-IDF + logistic regression", "Fine-tuned DistilBERT"])
comparison.round(3)

Be as honest here as on Day 3. On short, fairly formulaic messages like these, TF-IDF is often close to a fine-tuned
transformer, sometimes level with it, and it trains in about a second on any laptop. The transformer tends to pull
ahead when:

- meaning depends on word order or negation (*"not paid"* vs *"paid, not late"*),
- messages are longer and mix several topics,
- there's little labelled data, because the pre-trained model already knows the language,
- the vocabulary varies a lot (spelling variants, mixed languages), because sub-word tokens cope with words never seen
  in training.

Whatever the result on your run, the question for production is the same as Day 2's: is the gain worth the extra
cost of running and maintaining a 540 MB model instead of a 2 MB one?

### Where does it get confused?

A **confusion matrix** shows, for each actual category (rows), how the model's predictions (columns) were spread. The
diagonal, from top left to bottom right, is the correct predictions. Any other cell is a mix-up: the number in row
"delayed_benefit", column "pension_payment" is how many delayed-benefit messages were predicted as pension payments.

In [ ]:
# Rows = actual category, columns = predicted category.
cm = confusion_matrix(test["label"], bert_pred)
pd.DataFrame(cm, index=labels, columns=labels)

The same table as a **heatmap** is easier to scan: darker squares mean more messages. `imshow` draws the colours, and
the loop writes each count in its square.

In [ ]:
# The same table as a heatmap.
plt.figure(figsize=(8, 6.5))
plt.imshow(cm, cmap="Blues")
plt.xticks(range(len(labels)), labels, rotation=45, ha="right")
plt.yticks(range(len(labels)), labels)
for row in range(len(labels)):
    for col in range(len(labels)):
        plt.text(col, row, cm[row, col], ha="center", va="center", fontsize=8)     # the count in each square
plt.xlabel("predicted")
plt.ylabel("actual")
plt.title("DistilBERT confusion matrix, test set")
plt.show()

The biggest off-diagonal cells are usually pairs the *people* labelling the data also mix up: "contribution not
posted" vs "employer not remitting" (often the same underlying problem, seen from two sides), or "delayed benefit"
vs "pension payment". That's a sign the categories overlap, and it's a conversation for the customer service
manager: should those two categories be one queue?

> **Where we are.** You've compared the transformer and TF-IDF on the same test messages, for quality and for cost, and
> seen which categories get mixed up, often the same ones people mix up. Now decide how the model should be used.

### Theory: confidence, calibration, and when a person should take over

The model's highest probability for a message is its **confidence**. A useful routing rule sends the message to a team
only when confidence is high, and to a person otherwise. That only works if the confidence means what it says.

A model is **well calibrated** if, among all messages where it says 0.9, it's right about 90% of the time. Many neural
networks aren't: they say 0.95 and are right 80% of the time. You check calibration by grouping predictions by
confidence and comparing with the real accuracy in each group, which is what the threshold table below does.

This pattern (*act automatically when confident, hand over to a person when not*) is called **human in the loop**.
You'll see it again: Day 5's review queue was one, Day 9's handbook assistant refuses when its sources don't answer the
question, and Day 10's governance pack asks where people stay in control.

## 7. Routing with a safety valve

A model that's unsure should say so. Send a message to a team automatically only when the model is confident, and
send everything else to a person for triage. The threshold decides the trade-off between automation and accuracy.

The model's **confidence** in a prediction is its highest probability. For each threshold, work out two things:

- **routed automatically:** the share of messages whose confidence is at or above the threshold;
- **accuracy of automatic routing:** of those messages, the share the model got right.

A threshold of 0.0 routes everything; a higher threshold routes fewer messages, but should get more of them right.

In [ ]:
# The model's confidence in each prediction = its highest probability.
confidence = probs.max(axis=1)
correct = bert_pred == test["label"].to_numpy()        # True where the prediction was right

rows = []
for threshold in [0.0, 0.5, 0.7, 0.8, 0.9, 0.95]:
    automatic = confidence >= threshold
    rows.append({
        "confidence threshold": threshold,
        "routed automatically": automatic.mean(),
        "accuracy of automatic routing": correct[automatic].mean(),
    })
pd.DataFrame(rows).round(3)

Notice how few messages fall below even a high threshold. Fine-tuned transformers are often **overconfident**: they
report 0.95 on messages they get wrong. Before relying on the confidence score, check it against real accuracy on
held-out data, as in the table above, and if the two disagree, **calibrate** it: adjust the scores so that "0.9" really
means right 9 times in 10. (scikit-learn's `CalibratedClassifierCV` does this for classical models; for transformers,
"temperature scaling" is the usual fix.)

Pick the threshold with the customer service manager: "we can route about this share of messages automatically with
this accuracy, and a person handles the rest". Then map categories to teams:

In [ ]:
# Which team handles each category.
TEAM = {
    "delayed_benefit": "Benefits & Claims",
    "pension_payment": "Benefits & Claims",
    "contribution_not_posted": "Contributions & Compliance",
    "employer_not_remitting": "Contributions & Compliance",
    "digital_services": "ICT Service Desk",
    "registration_details": "Member Services",
    "customer_service": "Branch Operations",
    "general_enquiry": "Call Centre",
    "compliment": "Customer Experience",
}
THRESHOLD = 0.8      # route automatically only if at least 80% confident

The routing function takes a list of new messages and, for each one, returns its predicted category, the confidence,
and where to send it. Step by step:

1. tokenise all the messages together. `padding=True` pads them to the same length, and `return_tensors="pt"` returns
   PyTorch tensors; `.to(model.device)` puts them on the same device (GPU or CPU) as the model;
2. switch the model to scoring mode (`eval()`), and run it without tracking gradients (`no_grad()`);
3. turn the logits into probabilities with softmax, and bring them back to the CPU as a NumPy array;
4. for each message, pick the best category and its probability; route to the team if confident enough, otherwise to
   human triage.

`model(**encoded)` passes the tokenizer's output (ids and mask) to the model as named arguments.

In [ ]:
def route(messages):
    """Predict each message's category and confidence; send it to a team only if the model is confident enough."""
    # Tokenise all the messages together, as PyTorch tensors, on the same device as the model.
    encoded = tok(messages, truncation=True, max_length=MAX_LEN, padding=True, return_tensors="pt")
    encoded = encoded.to(model.device)

    # Score them without training.
    model.eval()
    with torch.no_grad():
        logits = model(**encoded).logits.float()
    probabilities = torch.softmax(logits, dim=-1).cpu().numpy()

    # One row per message: category, confidence, and where it goes.
    rows = []
    for i in range(len(messages)):
        best = int(probabilities[i].argmax())
        category = id2label[best]
        confidence = float(probabilities[i].max())
        if confidence >= THRESHOLD:
            send_to = TEAM[category]
        else:
            send_to = "Human triage"
        rows.append({"message": messages[i], "category": category, "confidence": round(confidence, 3), "send_to": send_to})
    return pd.DataFrame(rows)

Try it on five new messages, in both languages:

In [ ]:
route([
    "Mwajiri wetu amefunga kampuni na hatujui kama alipeleka michango yetu.",
    "The app says my member number is invalid since yesterday.",
    "My mother's pension did not arrive this month and she is 81.",
    "Thank you, the Arusha branch staff were excellent today.",
    "Is it true that the penalty for late payment changed?",
])

**Your turn (1).** Write five messages of your own, the kind you've actually seen at the Fund, including at least
two in Swahili and one that mixes two problems. Route them. Where the model is wrong or unsure, is that a model
problem or a category problem?

In [ ]:
# Your code here

> **Where we are.** You have a routing function that sends confident predictions to the right team and everything else
> to human triage, with a threshold agreed in plain terms. The last step is to save the model so it can be used outside
> this notebook.

## 8. Saving the model

A fine-tuned model is a folder: weights, configuration and tokenizer. Save both parts together, or you won't be able
to use it: the model can't read text without its own tokenizer.

In [ ]:
trainer.save_model("models/complaint_router")        # weights and configuration
tok.save_pretrained("models/complaint_router")       # the matching tokenizer
print("saved to models/complaint_router")

To use it later, in another notebook or behind an API (Day 9 shows how), Hugging Face's `pipeline` loads both parts and
does the tokenising, scoring and softmax for you:

```python
from transformers import pipeline
router = pipeline("text-classification", model="models/complaint_router", top_k=None)
router("Michango yangu ya miezi mitatu haionekani.")
```

---

## Looking back, looking ahead

**What you did in this lab.** You fine-tuned a pre-trained multilingual transformer to route complaints into nine
categories: you saw how it tokenises English and Swahili, prepared the data as a PyTorch `Dataset`, trained with the
Hugging Face `Trainer`, compared it fairly with TF-IDF on quality and cost, read its confusion matrix, set a confidence
threshold for automatic routing, and saved the model with its tokenizer.

**The ideas to carry forward.**

- Transformers give each word a meaning that depends on its context, through attention.
- Pre-training plus fine-tuning lets a model learn a new task from a few thousand examples.
- Small learning rates and few epochs protect what the model already knows.
- A strong classical baseline can be close to a transformer, at a fraction of the cost.
- Route automatically only when confident; check that confidence means what it says.

**What happens next.** Lab 3 pulls structured facts out of free text, combining a pre-trained spaCy model with rules
you write yourself.

## Your turn (more)

2. **Less data.** Retrain both models on only 300 training messages (`train.sample(300, random_state=SEED)`). Which
   one suffers more? What does that tell you about when a pre-trained model is worth it?
3. **English-only model.** Swap `MODEL_NAME` for `distilbert-base-uncased` (English only) and retrain. How does it do
   on the Swahili messages in the test set compared with the multilingual model?

In [ ]:
# Your code here

## Suggested answer to (2)

Many people expect the transformer to win easily when data is scarce, because it arrives already knowing the language.
Run the cells below and check whether that's what happens here.

First the TF-IDF model on 300 messages:

In [ ]:
small = train.sample(300, random_state=SEED)

# TF-IDF on 300 messages. min_df=1: with so little data, keep words seen even once.
tfidf_small = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=1, sublinear_tf=True)),
    ("clf", LogisticRegression(max_iter=3000, class_weight="balanced")),
])
tfidf_small.fit(small["text"], small["label"])
small_tfidf_f1 = f1_score(test["label"], tfidf_small.predict(test["text"]), average="macro")
print("TF-IDF on 300 examples, macro F1:", round(small_tfidf_f1, 3))

Then a fresh DistilBERT (starting again from the downloaded model, not the one you just fine-tuned) on the same 300
messages:

In [ ]:
# A fresh DistilBERT, fine-tuned on the same 300 messages (more epochs, because there's so little data).
small_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=len(labels), id2label=id2label, label2id=label2id)

small_args = TrainingArguments(output_dir="bert_small", num_train_epochs=6, per_device_train_batch_size=16,
                               learning_rate=5e-5, warmup_steps=12, save_strategy="no", eval_strategy="no",
                               fp16=GPU, report_to="none", seed=SEED, logging_steps=100)
small_trainer = Trainer(model=small_model, args=small_args, train_dataset=FeedbackDataset(small),
                        data_collator=DataCollatorWithPadding(tok))
small_trainer.train()

# Predict the test set: the highest-scoring category for each message.
small_pred = small_trainer.predict(test_ds).predictions.argmax(axis=1)
print("DistilBERT on 300 examples, macro F1:", round(f1_score(test["label"], small_pred, average="macro"), 3))

On this data it usually **doesn't**: TF-IDF keeps most of its accuracy on 300 examples, while the fine-tuned transformer
drops sharply. About 33 examples per category is very little for updating 135 million parameters, and these messages
contain strong keywords that a bag-of-words model picks up from a handful of examples.

The honest lesson: "pre-trained models need less data" is true compared with training a network from scratch, not
necessarily compared with a good classical baseline. With a few hundred examples, try the simple model first. If you
need a transformer with very little labelled data, look at approaches built for it: freezing most layers, training
for more epochs with a smaller learning rate, or using a sentence-embedding model plus logistic regression (you'll
use sentence embeddings on Day 9).